# Wan 2.2 Image-to-Video — Official Repository (T4 Colab)

Built on the **official [`Wan-Video/Wan2.2`](https://github.com/Wan-Video/Wan2.2) GitHub repository** — native inference (`generate.py`), not `diffusers`. `AutoPipelineForImage2Video` is never used.

**There is exactly one cell below.** It clones the repo, installs dependencies (self-healing for Python 3.13 — see inside), downloads the T4-compatible model, pauses once to ask you to upload a product photo, generates the video (auto-retrying with a shorter clip if a free T4 runs out of memory), and downloads the result. No other cell, no `os.kill()`, no step that needs you to come back and run something else.

**Required action:** `Runtime -> Restart session and run all`. This guarantees a fully clean process before the cell runs — if you instead use `Runtime -> Restart session` on its own, nothing executes (that command only restarts; it doesn't run anything), which is indistinguishable from "nothing happened." **Run all** works too on a runtime you haven't used for anything else yet; **Restart session and run all** is the safe choice regardless.

**Model:** `Wan-AI/Wan2.2-TI2V-5B` via the official `ti2v-5B` task — the only Wan 2.2 checkpoint the repo positions for a single consumer GPU (the 14B tasks need 80GB+ VRAM per the official README). Output size is `704*1280`, `ti2v-5B`'s only 9:16 option (`480*832` is not valid for this task and raises `AssertionError` if forced — verified directly against `wan/configs/__init__.py`).

In [ ]:
import os
import re
import subprocess

def step(msg):
    print(f'\n=== {msg} ===')

# ---------------------------------------------------------------------
# 1) Confirm the GPU
# ---------------------------------------------------------------------
step('1/7 Confirming GPU')
subprocess.run(['nvidia-smi'])

# ---------------------------------------------------------------------
# 2) Clone the official repository
# ---------------------------------------------------------------------
step('2/7 Cloning the official Wan-Video/Wan2.2 repository')
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/Wan2.2'])
clone = subprocess.run(
    ['git', 'clone', '--quiet', 'https://github.com/Wan-Video/Wan2.2.git'],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    print(clone.stderr[-2000:])
    raise RuntimeError('git clone failed \u2014 check your network/GitHub availability and re-run this cell.')
os.chdir('/content/Wan2.2')
print('Repository ready at /content/Wan2.2')

# ---------------------------------------------------------------------
# 3) Install dependencies (self-healing for Python 3.13)
# ---------------------------------------------------------------------
step('3/7 Installing dependencies')
EXCLUDE_PREFIXES = ('flash_attn', 'torch', 'torchvision', 'torchaudio')
with open('requirements.txt') as f:
    pinned_specs = [
        line.strip() for line in f
        if line.strip() and not line.strip().startswith('#')
        and not any(line.strip().lower().startswith(p) for p in EXCLUDE_PREFIXES)
    ]
unpinned_specs = [re.split(r'[<>=!~\[]', spec, 1)[0].strip() for spec in pinned_specs]

def pip_install(specs, label):
    print(f'  {label}: pip install {" ".join(specs)}')
    return subprocess.run(['pip', 'install', '-q'] + specs, capture_output=True, text=True)

result = pip_install(pinned_specs, 'Attempt 1/2 (pinned versions from requirements.txt)')
if result.returncode != 0:
    print('  Pinned install failed (likely no Python 3.13 wheel for one of them):')
    print('  ' + (result.stderr or result.stdout or '')[-1500:])
    print('  Automatically retrying unpinned so pip resolves current, compatible releases...')
    result = pip_install(unpinned_specs, 'Attempt 2/2 (unpinned fallback)')
    if result.returncode != 0:
        print((result.stderr or result.stdout or '')[-1500:])
        raise RuntimeError('Dependency install failed on both attempts \u2014 see the pip error above.')

hf_result = subprocess.run(['pip', 'install', '-q', 'huggingface_hub[cli]'], capture_output=True, text=True)
if hf_result.returncode != 0:
    print(hf_result.stderr[-1500:])
    raise RuntimeError('Failed to install huggingface_hub[cli].')
print('Dependencies installed.')

# ---------------------------------------------------------------------
# 4) Verify the environment \u2014 the one place a stale-import problem
#    (the kind a runtime restart fixes) would actually show up
# ---------------------------------------------------------------------
step('4/7 Verifying the environment')
try:
    import torch
    print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError(
            'No GPU detected. Click Runtime > Change runtime type, select T4 GPU, save, '
            'then Runtime > Restart session and run all.'
        )
    import wan  # noqa: F401 (sanity check only)
    print('wan package import: OK')
except Exception as e:
    message = str(e).lower()
    looks_like_stale_import = any(
        s in message for s in ('numpy', 'binary incompat', 'dtype size changed', 'symbol not found')
    )
    if looks_like_stale_import:
        raise RuntimeError(
            f'{e}\n\n'
            'This specific error means a package already loaded in memory from an earlier run '
            'conflicts with what was just installed \u2014 the one situation this notebook cannot '
            'fix by itself. Click Runtime > Restart session and run all (this always starts a '
            'fully clean process) and this cell will succeed on that run.'
        )
    raise
print('Environment ready \u2014 no restart required.')

# ---------------------------------------------------------------------
# 5) Download the T4-compatible model (TI2V-5B)
# ---------------------------------------------------------------------
step('5/7 Downloading Wan-AI/Wan2.2-TI2V-5B (several GB \u2014 first run takes a few minutes)')
download = subprocess.run(
    ['huggingface-cli', 'download', 'Wan-AI/Wan2.2-TI2V-5B', '--local-dir', './Wan2.2-TI2V-5B'],
    capture_output=True, text=True,
)
if download.returncode != 0:
    print(download.stderr[-1500:])
    raise RuntimeError('Model download failed \u2014 check the Hugging Face status above and re-run this cell.')
print('Model ready.')

# ---------------------------------------------------------------------
# 6) Upload your product image \u2014 the only manual step
# ---------------------------------------------------------------------
step('6/7 Waiting for you to upload one product photo')
from google.colab import files
uploaded = files.upload()
image_filename = next(iter(uploaded))
image_path = f'/content/Wan2.2/{image_filename}'
print(f'Uploaded: {image_filename}')

# ---------------------------------------------------------------------
# 7) Generate a 9:16 video (auto-retries on out-of-memory), then
#    preview and automatically download it
# ---------------------------------------------------------------------
step('7/7 Generating the video')

SIZE = '704*1280'  # ti2v-5B's only 9:16 option \u2014 480*832 is not valid for this task
FPS = 24
DURATION_LADDER_SECONDS = [5, 3, 2, 1]  # automatic fallback order if one length hits CUDA OOM
PROMPT = 'A realistic, premium commercial product video. Natural, smooth camera motion, cinematic lighting.'  # edit me
SAVE_FILE = 'output.mp4'

def frame_num_for(seconds):
    return int(round((FPS * seconds - 1) / 4)) * 4 + 1  # frame counts must be 4n+1

def run_generate(seconds):
    frame_num = frame_num_for(seconds)
    print(f'  Trying {frame_num} frames (~{frame_num / FPS:.1f}s @ {FPS}fps) at {SIZE}...')
    return subprocess.run(
        [
            'python', 'generate.py',
            '--task', 'ti2v-5B',
            '--size', SIZE,
            '--ckpt_dir', './Wan2.2-TI2V-5B',
            '--offload_model', 'True',
            '--convert_model_dtype',
            '--t5_cpu',
            '--image', image_path,
            '--prompt', PROMPT,
            '--frame_num', str(frame_num),
            '--save_file', SAVE_FILE,
        ],
        capture_output=True, text=True,
    )

succeeded = False
for attempt_index, seconds in enumerate(DURATION_LADDER_SECONDS):
    proc = run_generate(seconds)
    if proc.returncode == 0:
        print(f'Generated a ~{seconds}s clip: {SAVE_FILE}')
        succeeded = True
        break
    stderr_tail = (proc.stderr or '')[-3000:]
    is_oom = 'out of memory' in stderr_tail.lower()
    is_last_attempt = attempt_index == len(DURATION_LADDER_SECONDS) - 1
    if is_oom and not is_last_attempt:
        print(f'  CUDA out of memory at ~{seconds}s \u2014 automatically retrying with a shorter clip...')
        continue
    print(stderr_tail)
    reason = 'CUDA out of memory even at the shortest fallback length' if is_oom else 'generate.py failed'
    raise RuntimeError(
        f'{reason}. This T4 session may have less free VRAM than usual right now \u2014 '
        'Runtime > Disconnect and delete runtime, then Runtime > Restart session and run all '
        'often gets a session with more headroom. Full error above.'
    )

if not succeeded:
    raise RuntimeError('Video generation did not succeed.')

from IPython.display import Video, display
display(Video(SAVE_FILE, embed=True))
files.download('output.mp4')
print('\n\u2705 All done \u2014 output.mp4 has been downloaded.')